In [2]:
import requests
import json

url = "https://10.1.13.168:8443/mbdevai/openai/deployments/azure-chat-engine-gpt-4o-base/chat/completions?api-version=2024-02-15-preview&api-key=REPLACE_WITH_LOCAL_SECRET"

payload = json.dumps({
  "stream": False,
  "messages": [
    {
      "role": "user",
      "content": "viết giùm tôi một câu truyện kinh dị về thầy trừ tà Minh Quỷ, hắn là trường phái nửa chính nửa tà"
    }
  ],
  "temperature": 0
})
headers = {
  'Content-Type': 'application/json'
}

response = requests.request("POST", url, headers=headers, data=payload, verify=False)

print(response.text)


c:\Users\thanglq2\AppData\Local\anaconda3\envs\beegen\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.1.13.168'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{"choices":[{"content_filter_results":{"hate":{"filtered":false,"severity":"safe"},"protected_material_code":{"filtered":false,"detected":false},"protected_material_text":{"filtered":false,"detected":false},"self_harm":{"filtered":false,"severity":"safe"},"sexual":{"filtered":false,"severity":"safe"},"violence":{"filtered":false,"severity":"safe"}},"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"Trong một ngôi làng nhỏ nằm sâu trong rừng rậm, có một người đàn ông tên Minh Quỷ. Hắn là một thầy trừ tà nổi tiếng, nhưng không ai biết rõ về nguồn gốc và quá khứ của hắn. Minh Quỷ thuộc trường phái nửa chính nửa tà, sử dụng cả phép thuật trắng và đen để đối phó với những thế lực siêu nhiên.\n\nMột đêm nọ, một gia đình trong làng bị ám bởi một linh hồn độc ác. Những tiếng khóc than và tiếng cười man rợ vang lên từ ngôi nhà, khiến cả làng không ai dám lại gần. Người dân quyết định tìm đến Minh Quỷ để nhờ giúp đỡ.\n\nMinh Quỷ đến ngôi nhà bị ám, mang theo một chiếc túi đầy

In [ ]:
import time
import asyncio
import aiohttp
import ssl
from typing import List

async def test_shared_session(prompts: List[str], batch_size: int = 10) -> float:
    """Test shared session with controlled concurrency"""
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    start_time = time.time()
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=batch_size)) as session:
        # Process in batches to avoid overwhelming the connection
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i + batch_size]
            tasks = [
                make_vllm_async_request(prompt, session, ssl_context)
                for prompt in batch
            ]
            await asyncio.gather(*tasks)
    return time.time() - start_time

async def test_new_sessions(prompts: List[str], batch_size: int = 10) -> float:
    """Test new sessions with controlled concurrency"""
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    start_time = time.time()
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        tasks = [
            make_vllm_async_request_new_session(prompt, ssl_context)
            for prompt in batch
        ]
        await asyncio.gather(*tasks)
    return time.time() - start_time

async def make_vllm_async_request(prompt: str, session: aiohttp.ClientSession, ssl_context: ssl.SSLContext) -> str:  
    url = "http://10.1.39.127:30042/v1/chat/completions"
    payload = {
        "model": "model/beegen-model-v1",
        "temperature": 0,
        "messages": [{"role": "user", "content": prompt}]
    }
    headers = {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer mb123456789'
    }
    
    try:
        async with session.post(url, json=payload, headers=headers, ssl=ssl_context) as response:
            result = await response.json()
            return result['choices'][0]['message']['content']
    except Exception as e:
        print(f"Error in shared session: {str(e)}")
        raise

async def make_vllm_async_request_new_session(prompt: str, ssl_context: ssl.SSLContext) -> str:  
    url = "http://10.1.39.127:30042/v1/chat/completions"
    payload = {
        "model": "model/beegen-model-v1",
        "temperature": 0,
        "messages": [{"role": "user", "content": prompt}]
    }
    headers = {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer mb123456789'
    }
    
    try:
        async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=1)) as session:
            async with session.post(url, json=payload, headers=headers, ssl=ssl_context) as response:
                result = await response.json()
                return result['choices'][0]['message']['content']
    except Exception as e:
        print(f"Error in new session: {str(e)}")
        raise
    

import random


async def run_detailed_performance_test():
    # Test with different batch sizes
    for num_requests in [20, 50]:
        for batch_size in [5, 10, 20]:
            print(f"\nTesting with {num_requests} total requests, batch size {batch_size}:")
            prompts = [f"What is {random.randint(1, 100)}+{random.randint(1, 100)}?" for i in range(num_requests)]
            
            # Test new sessions
            print("Testing new sessions...")
            new_sessions_time = await test_new_sessions(prompts, batch_size)
            print(f"New sessions time: {new_sessions_time:.3f} seconds")
            
            # Small delay between tests
            await asyncio.sleep(1)
            
            # Test shared session
            print("Testing shared session...")
            shared_time = await test_shared_session(prompts, batch_size)
            print(f"Shared session time: {shared_time:.3f} seconds")
            
            # Calculate difference
            if shared_time > new_sessions_time:
                diff_percent = ((shared_time - new_sessions_time) / new_sessions_time) * 100
                print(f"Shared session is {diff_percent:.1f}% slower than new sessions")
            else:
                diff_percent = ((new_sessions_time - shared_time) / shared_time) * 100
                print(f"New sessions is {diff_percent:.1f}% slower than shared session")

# Run the test
asyncio.run(run_detailed_performance_test())

In [21]:
import asyncio
import aiohttp
import json
import ssl
import nest_asyncio
from typing import List
import uuid

# Enable asyncio in Jupyter
nest_asyncio.apply()

async def make_async_request(prompt: str, session: aiohttp.ClientSession, ssl_context: ssl.SSLContext) -> str:
    print(f"request prompt: {uuid.uuid4()}")
    # await asyncio.sleep(0.00001)  # 100ms delay
    url = "https://10.1.13.168:8443/mbdevai/openai/deployments/azure-chat-engine-gpt-4o-base/chat/completions?api-version=2024-02-15-preview&api-key=REPLACE_WITH_LOCAL_SECRET"
    
    payload = {
        "stream": False,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    async with session.post(url, json=payload, ssl=ssl_context) as response:
        result = await response.json()
        return result['choices'][0]['message']['content']
    

async def make_vllm_async_request(prompt: str, session: aiohttp.ClientSession, ssl_context: ssl.SSLContext) -> str:  
    print(f"Making connect with prompt: {prompt}")
    url = "http://10.1.39.127:30042/v1/chat/completions"
    
    payload = {
        "model": "model/beegen-model-v1",
        "temperature": 0,
        "top_p": 0.001,
        "n": 1,
        "stream": False,
        "frequency_penalty": 0,
        "presence_penalty": 0,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
    
    headers = {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer mb123456789',
    }

    async with session.post(url, json=payload, headers=headers, ssl=ssl_context) as response:
        result = await response.json()
        return result['choices'][0]['message']['content']


async def make_multiple_requests(prompts: List[str]) -> List[str]:
    # Create SSL context
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    async with aiohttp.ClientSession() as session:
        tasks = [
            make_vllm_async_request(prompt, session, ssl_context)
            for prompt in prompts
        ]
        return await asyncio.gather(*tasks)

# Usage example:
prompts = [f"What is {i}+{i}?" for i in range(10)]
results = asyncio.run(make_multiple_requests(prompts))
results

Making connect with prompt: What is 0+0?
Making connect with prompt: What is 1+1?
Making connect with prompt: What is 2+2?
Making connect with prompt: What is 3+3?
Making connect with prompt: What is 4+4?
Making connect with prompt: What is 5+5?
Making connect with prompt: What is 6+6?
Making connect with prompt: What is 7+7?
Making connect with prompt: What is 8+8?
Making connect with prompt: What is 9+9?


['0 + 0 equals 0.',
 '1+1 equals 2.',
 '2 + 2 equals 4.',
 '3 + 3 equals 6.',
 '4 + 4 equals 8.',
 '5 + 5 equals 10.',
 '6 + 6 equals 12.',
 '7 + 7 equals 14.',
 '8 + 8 equals 16.',
 '9 + 9 equals 18.']

In [4]:
query = "đầu pin Thẻ Visa Platinum là bao nhiêu"
    
documents = [
    "Tên tài liệu chứa chunk: 66. Thẻ ghi nợ quốc tế Visa Debit\n Nội dung chunk: KH sử dụng thẻ để giao tại các Đơn vị chấp nhận thẻ và ATM có hỗ trợ thẻ VISA tại Việt Nam và trên toàn cầu. 2, DẤU HIỆU NHẬN BIẾT. 2.1 Thông tin trên mặt thẻ:. - Mặt trước thẻ:. + Logo Ngân hàng TMCP Quân đội. + Logo TCTQT Visa. + Số thẻ. + Ngày hiệu lực của thẻ. + Tên chủ thẻ. - Mặt sau thẻ. + Số điện thoại trung tâm dịch vụ khách hàng MB 247;. + Dải băng từ;. + Hologram của Visa;. + CVV2 (Sử dụng trong các giao dịch trực tuyến).. + Dải chữ ký.. 2.2 Nhận diện thẻ. - Là thẻ chip. - Dải BIN nhận biết:. + Hạng classic: 408904. + Hạng Platinum: 408905. - Thời hạn sử dụng: 05 năm kể từ ngày phát hành trên hệ thống. 2.3 Quy định về mã PIN:. - PIN gồm 4 chữ số. - Sai PIN tối đa 3 lần, trường hợp sai PIN quá 3 lần, thẻ sẽ bị khóa tạm thời.. - PIN được bắt buộc sử dụng khi giao dịch trên ATM, chuyển khoản trên POS.. - Không bắt buộc đổi PIN lần đầu/Đổi PIN trên ATM MB. 3, ĐỐI TƯỢNG VÀ ĐIỀU KIỆN PHÁT HÀNH. 3.1 Thẻ Visa debit hạng Platinum. - Đối tượng: Tất cả KH đủ điều kiện PH thẻ ghi nợ và/hoặc đang sử dụng thẻ Visa Debit của MB. - Kênh phát hành:. + Thẻ phi vật lý: .. Nếu có nhu cầu KH chỉ phát hành được thẻ phi vật lý thẻ Visa Debit hạng Platinum trên App MBBank.",
    "Tên tài liệu chứa chunk: 67. Thẻ tín dụng quốc tế Visa Credit\n Nội dung chunk: -\tThẻ tín dụng quốc tế Visa/JCB Platinum. -\tThẻ tín dụng quốc tế Visa/JCB Modern Youth. \tKênh phát hành: App MB. \tHạn mức thẻ: Thông hạn mức, nhỏ hơn hoặc bằng hạn mức thẻ gốc KH đang có, tối thiểu 1,000,000 VND và tối đa 100,000,000 VND. \tTính năng thẻ:. \tTương tự thẻ tín dụng phát hành trên App MB hiện tại, ngoại trừ chưa hỗ trợ in thẻ vật lý. \tKH có thể điều chỉnh hạn mức thẻ thêm (nhỏ hơn hạn mức hiện tại) tại Dịch vụ thẻ trên App MB. 5, TÍNH NĂNG SẢN PHẨM. 5.1 Trên ATM/CRM. -\tRút tiền mặt. -\tĐổi PIN ( Chỉ áp dụng với ATM MB ). -\tTruy vấn hạn mức khả dụng. -\tSao kê 05 giao dịch gần nhất. 5.2 Trên POS. -\tThanh toán hàng hóa dịch vụ. -\tỨng tiền mặt (Tại 1 số đơn vị đặc thù => không nên tư vấn với KH). -\tThanh toán hóa đơn. 5.3 Trên Internet. 5.3.1 Thanh toán hàng hóa, dịch vụ (Thanh toán ecom - Thanh toán online) (trong và ngoài nước).. Giao dịch này không cần nhập PIN, tuy nhiên khi thực hiện giao dịch thanh toán online, 1 số đơn vị chấp nhận thẻ (ĐVCNT) (App/Web) sẽ yêu cầu nhập thêm OTP (3D Secure).. Việc có nhập OTP hay không phụ thuộc vào quy định của ĐVCNT, không phải do quy định tại MB.",
    "Tên tài liệu chứa chunk: 67. Thẻ tín dụng quốc tế Visa Credit\n Nội dung chunk: \tẢnh chụp thẻ hội viên. \tẢnh chụp thông tin thẻ hội viên trên App của Bamboo/VietNam Airlines/Vietravel, có xác minh của đối tác.. Bao gồm các thông tin: Tên, số hội viên, hạng hội viên, ngày hết hạn, số giấy tờ tùy thân). \tCIC còn hiệu lực. \tTờ trình phát hành thẻ (Do Chi nhánh thực hiện). f, Quy trình phát hành: CN/PGD: Thu thập hồ sơ của KH và đẩy lên hệ thống để phát hành/ Trung tâm VHST: Phát hành và chuyển phát thẻ cho KH theo địa chỉ đăng ký. 4, QUY ĐỊNH VỀ THẺ CHÍNH, THẺ PHỤ:. 4.1 Số lượng phát hành:. -\tĐối với các sản phẩm thẻ tín dụng quốc tế MB, mỗi KH chỉ được phát hành 01 thẻ chính trên hệ thống MB.. -\tĐối với hạng thẻ Classic và Gold: Mỗi thẻ chính được phép phát hành tối đa 08 thẻ phụ.. -\tĐối với hạng thẻ Platinum: Mỗi thẻ chính được phép phát hành tối đa 02 thẻ phụ.. -\tMỗi thẻ chính có thể được phát hành thẻ phụ cùng hạng.. Thẻ phụ có hạn mức thấp hơn hoặc bằng hạn mức tín dụng của thẻ chính.. Hạn mức của thẻ phụ được MB cấp căn cứ trên cơ sở đề nghị của chủ thẻ chính và quyết định của MB.. 4.2 Quyền hạn, trách nhiệm của chủ thẻ chính, chủ thẻ phụ. \tChủ thẻ chính:. Là người có quyền đưa ra tất cả các yêu cầu liên quan đến việc phát hành và sử dụng của thẻ chính và thẻ phụ mà không cần sự đồng ý của chủ thẻ phụ.. Chịu trách nhiệm thanh toán đối với toàn bộ các giao dịch, lãi và phí của chủ thẻ chính và chủ thẻ phụ phát sinh trên tài khoản thẻ tín dụng.. \tChủ thẻ phụ:. Được phép đưa ra các yêu cầu liên quan trực tiếp đến việc sử dụng thẻ phụ đó, bao gồm: khóa thẻ tạm thời, khóa thẻ vĩnh viễn, in lại PIN, mở khóa thẻ (trừ khi thẻ phụ bị khóa do yêu cầu của chủ thẻ chính).. Chịu trách nhiệm về việc sử dụng thẻ phụ với chủ thẻ chính.. Lưu ý: Đối với trường hợp chủ thẻ chính yêu cầu khóa thẻ, chủ thẻ phụ không được yêu cầu mở khóa thẻ hoặc in lại PIN.. 4.3 Quy định về việc phát hành thẻ thêm trên App MBBANK. \tMB cho phép KH phát hành thêm thẻ tín dụng với hạn mức tùy chọn (nhỏ hơn hoặc bằng hạn mức của thẻ đã được cấp trước đó) trên App MB đối với các Khách hàng (KH) đang có thẻ tín dụng MB.. \tPhù hợp sử dụng chi tiêu online, add nguồn thanh toán.. \tKhách hàng áp dụng: Áp dụng cho các KH có App MB và đang sử dụng thẻ tín dụng MB với hạn mức từ 1,000,000 VND trở lên.. \tSản phẩm thẻ áp dụng: KH được phát hành thẻ thêm với hạng thẻ bằng với thẻ gốc đang có, hoặc thẻ Visa/JCB Modern Youth.. Các loại thẻ cho phép phát hành thẻ thêm bao gồm:. -\tThẻ tín dụng quốc tế Visa Infinite. -\tThẻ tín dụng quốc tế Visa/JCB Priority Platinum (theo tiến độ triển khai sản phẩm JCB Priority). -\tThẻ tín dụng quốc tế Visa/JCB Platinum. -\tThẻ tín dụng quốc tế Visa/JCB Modern Youth. \tKênh phát hành: App MB. \tHạn mức thẻ: Thông hạn mức, nhỏ hơn hoặc bằng hạn mức thẻ gốc KH đang có, tối thiểu 1,000,000 VND và tối đa 100,000,000 VND. \tTính năng thẻ:. \tTương tự thẻ tín dụng phát hành trên App MB hiện tại, ngoại trừ chưa hỗ trợ in thẻ vật lý. \tKH có thể điều chỉnh hạn mức thẻ thêm (nhỏ hơn hạn mức hiện tại) tại Dịch vụ thẻ trên App MB. 5, TÍNH NĂNG SẢN PHẨM. 5.1 Trên ATM/CRM. -\tRút tiền mặt. -\tĐổi PIN ( Chỉ áp dụng với ATM MB ). -\tTruy vấn hạn mức khả dụng",
    "Tên tài liệu chứa chunk: 67. Thẻ tín dụng quốc tế Visa Credit\n Nội dung chunk: Logo Ngân hàng TMCP Quân đội. Logo của TCTQT. Số thẻ dập nổi: Gồm 16 số in dập nổi, đều, thẳng hàng. Thời gian hiệu lực thẻ. Tên chủ thẻ. Dải băng từ: Dải băng từ phải thẳng, mịn, không được tẩy xóa.. Chủ thẻ không được ký lên dải băng từ. Dải chữ ký: Lưu chữ ký của chủ thẻ, có mã số bảo mật in bên cạnh. Mã số bảo mật CVV2: 3 ký tự số được in ở cạnh dải chữ ký. 2.2 Nhận diện thẻ. - Dải BIN nhận biết:. + Thẻ Visa Classic: 484803. + Thẻ Visa Gold: 484804. + Thẻ Visa Platinum: 472674. - Thời hạn sử dụng: Thời hạn hiệu lực thẻ: 05 năm. 3, ĐỐI TƯỢNG VÀ ĐIỀU KIỆN PHÁT HÀNH:. 3.1 Đối tượng: Cá nhân người cư trú (là người Việt Nam hoặc người nước ngoài), có nhu cầu sử dụng thẻ tín dụng, đáp ứng các điều kiện về phát hành và sử dụng thẻ theo quy định của pháp luật và Quy định của MB và các quy định nội bộ khác có liên quan.. 3.2 Điều kiện và hồ sơ phát hành:. 3.2.1 Chính sách phát hành thẻ tín dụng cho nhóm KH đã được cấp hạn mức cho vay có tài sản bảo đảm. (Văn bản tham chiếu: Thông báo số 56/TB-136 ngày 09/09/2023 của Khối NHS)",
    "Tên tài liệu chứa chunk: 66. Thẻ ghi nợ quốc tế Visa Debit\n Nội dung chunk: - Cấp lại mã PIN: KH soạn tin theo cú pháp: MB PIN SốCMT 4Sốđầuthẻ 4sốcuốithẻ gửi 8136. - Trong đó:. + MOKHOATHE: Mở khóa thẻ, viết liền không dấu. + KHOATHE: Khóa thẻ, viết liền không dấu. + MO3D: Mở 3D, viết liền không dấu. + MOINTERNET: Mở internet, viết liền không dấu. + DONGINTERNET: Đóng internet, viết liền không dấu. + 4Sốđầuthẻ: 4 số đầu tiên trong dãy số in nổi trên mặt thẻ. + 4sốcuốithẻ: 4 số cuối cùng trong dãy số in nổi trên mặt thẻ. + SốCMT: Số giấy tờ tùy thân KH đăng ký với MB. 5, HẠN MỨC SỬ DỤNG. 5.1 Đối với thẻ Visa debit hạng classic. - Hạn mức rút tiền tối đa một giao dịch: 20 triệu đồng. - Hạn mức rút tiền tối đa/ngày: Trong nước: 50 triệu đồng ; Nước ngoài: 30 triệu đồng hoặc ngoại tệ quy đổi tương đương.. - Hạn mức chuyển khoản qua ATM một lần: 50 triệu đồng. - Hạn mức chuyển khoản qua ATM một ngày: hạng Classic: 50 triệu đồng. - Hạn mức thanh toán một lần trên POS hoặc Internet: hạng Classic: 50 triệu đồng. - Hạn mức thanh toán một ngày trên POS hoặc Internet: hạng Classic: 50 triệu đồng. - Số lần giao dịch một ngày: 20 lần. 5.2 Đối với thẻ Visa debit hạng Platinum. - Hạn mức rút tiền tối đa một giao dịch: 20 triệu đồng. - Hạn mức rút tiền tối đa một ngày: 100 triệu đồng (đối với giao dịch trong nước), 30 triệu đồng hoặc ngoại tệ quy đổi tương đương (đối với giao dịch tại nước ngoài). - Hạn mức chuyển khoản qua ATM một lần: 100 triệu đồng. - Hạn mức chuyển khoản qua ATM một ngày: 100 triệu đồng. - Hạn mức thanh toán một lần trên POS hoặc Internet: 100 triệu đồng. - Hạn mức thanh toán một ngày trên POS hoặc Internet: 100 triệu đồng. - Số lần giao dịch một ngày: 20 lần. Lưu ý: KH có thể yêu cầu điều chỉnh hạn mức theo nhu cầu bằng cách gửi yêu cầu trợ giúp tại CN/PGD MB hoặc qua các kênh hỗ trợ khác theo quy định của MB từng thời kỳ (trừ hạn mức rút tiền tại nước ngoài). 6, QUY ĐỊNH VỀ PHÍ. Nguyên tắc thu phí:. - Phí của thẻ được trừ từ TKTT của KH. - Mức phí phụ thuộc từng loại sản phẩm & đối tượng khách hàng sẽ có các mức phí khác nhau. - Đối với phí phát hành thẻ: Thu thời điểm phát hành thẻ. - Đối với phí thường niên/phí duy trì thẻ: Thu phí theo năm, 1 năm/lần và phí thu trước.. - Đối với phí giao dịch:. + Bao gồm: Phí rút tiền; Phí chuyển khoản; phí truy vấn số dư…. + Thu vào thời điểm KH thực hiện giao dich.. + Phí chuyển đổi ngoại tệ (Áp dụng đối với các GD bằng ngoại tệ).",
    "Câu hỏi: Đặc điểm Pin của thẻ visa debit?\nCâu trả lời: PIN gồm 4 chữ số\nPIN được bắt buộc sử dụng khi giao dịch trên ATM, chuyển khoản trên POS.\nKhông bắt buộc đổi PIN lần đầu/Đổi PIN trên ATM MB",
    "Câu hỏi: Phí thường niên thẻ tín dụng Visa, thẻ tín dụng quốc tế JCB hoặc thẻ tín dụng Visa Platinum Priority là bao nhiêu?\nCâu trả lời: Phí thường niên của thẻ tín dụng Visa, JCB, hoặc Visa Platinum Priority theo biểu phí hoặc chính sách thẻ của MB quy định từng thời kỳ. Chính sách hiện tại: Miễn phí (với thẻ được cấp hạn mức). Tham khảo biểu phí tại đây.",
    "Câu hỏi: Phí hủy thẻ tín dụng Visa Platinum Priority là bao nhiêu?\nCâu trả lời: Phí hủy thẻ tín dụng Visa Platinum Priority tuân theo biểu phí của MB quy định từng thời kỳ. Tham khảo biểu phí tại đây.",
    "Câu hỏi: Phí chuyển đổi ngoại tệ áp dụng cho thẻ Visa Platinum Priority?\nCâu trả lời: Phí được áp dụng theo biểu phí hoặc theo chính sách Quy định của MB từng thời, tham khảo biểu phí tại đây: https://confluence.mbbank.com.vn/pages/viewpage.action?pageId=119016762",
    "Câu hỏi: Phí thường niên thẻ tín dụng quốc tế JCB (bao gồm các hạng thẻ Classic, Gold, Platinum) là bao nhiêu?\nCâu trả lời: Theo biểu phí hoặc chính sách thẻ của MB được quy định từng thời kỳ. Chính sách thời điểm hiện tại: Miễn Phí (với thẻ được cấp hạn mức)\nVui lòng tham khảo biểu phí tại đây https://confluence.mbbank.com.vn/pages/viewpage.action?pageId=287463752\n"
]

classify_prompt = """Đọc tài liệu sau và trả lời CÓ hoặc KHÔNG, kèm lý do ngắn gọn:
<DOCUMENT>{document}</DOCUMENT>

Tài liệu trên có chứa thông tin trực tiếp để trả lời câu hỏi: "{question}" không?

Format trả lời:
Đáp án: [CÓ/KHÔNG]
Lý do: [1 câu giải thích ngắn gọn trong 20 từ]"""


prompts = [
    classify_prompt.format(document=doc, question=query)
    for doc in documents
]

In [5]:
await make_multiple_requests(prompts=prompts)

['Đáp án: CÓ\nLý do: Tài liệu cung cấp dải BIN nhận biết cho hạng Platinum là 408905, đây chính là đầu số của thẻ.',
 'Đáp án: KHÔNG\nLý do: Tài liệu không đề cập đến đầu pin cụ thể của Thẻ Visa Platinum.',
 'Đáp án: KHÔNG\nLý do: Tài liệu không đề cập đến thông tin về đầu pin của Thẻ Visa Platinum.',
 'Đáp án: CÓ\nLý do: Tài liệu cung cấp BIN (dải số nhận biết) cho Thẻ Visa Platinum là 472674.',
 'Đáp án: KHÔNG\nLý do: Tài liệu không đề cập đến "đầu pin" của Thẻ Visa Platinum.',
 'Đáp án: KHÔNG\nLý do: Tài liệu chỉ nêu số chữ số của PIN và cách sử dụng, không đề cập đến đầu số cụ thể của Thẻ Visa Platinum.',
 'Đáp án: KHÔNG\nLý do: Tài liệu không đề cập đến "đầu pin" của thẻ Visa Platinum.',
 'Đáp án: KHÔNG\nLý do: Tài liệu không đề cập đến phí thường niên (đầu pin) của thẻ Visa Platinum.',
 'Đáp án: KHÔNG\nLý do: Tài liệu không đề cập đến thông tin về đầu pin của Thẻ Visa Platinum.',
 'Đáp án: KHÔNG\nLý do: Tài liệu chỉ đề cập đến phí thường niên của thẻ JCB, không có thông tin về đầ